### Game Setup: Casper's Kitchen Rescue

This stage creates the game schema, seeds mystery data anomalies into the existing
Casper's data, and creates the quest state and answer tables that power the quest
controller app.

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create game schema and tables

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.game")
print(f"\u2705 Created schema {CATALOG}.game")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_state (
  player_id STRING,
  level INT,
  status STRING,
  answer STRING,
  correct BOOLEAN,
  started_at TIMESTAMP,
  completed_at TIMESTAMP,
  hints_used INT,
  score INT
)
COMMENT 'Tracks player progress through Casper Kitchen Rescue quest levels'
""")
print(f"\u2705 Created {CATALOG}.game.quest_state")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_answers (
  level INT,
  question_key STRING,
  acceptable_answers ARRAY<STRING>,
  hint STRING,
  max_score INT
)
COMMENT 'Hidden answer key for quest validation - do not expose to players'
""")
print(f"\u2705 Created {CATALOG}.game.quest_answers")

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.leaderboard (
  player_id STRING,
  player_name STRING,
  total_score INT,
  levels_completed INT,
  total_hints_used INT,
  started_at TIMESTAMP,
  finished_at TIMESTAMP
)
COMMENT 'Leaderboard for Casper Kitchen Rescue'
""")
print(f"\u2705 Created {CATALOG}.game.leaderboard")

##### Seed mystery anomalies

We create views that overlay detectable anomalies on the existing gold tables.
These give players something concrete to discover in Level 1 (Genie) and Level 2 (Dashboard).

In [ ]:
# Determine which location has the most orders (will become the "sabotaged" one)
# and which brand is the biggest revenue driver there
anomaly_location = spark.sql(f"""
  SELECT location_id, COUNT(*) as cnt
  FROM {CATALOG}.lakeflow.gold_order_header
  GROUP BY location_id
  ORDER BY cnt DESC
  LIMIT 1
""").collect()

if anomaly_location:
    ANOMALY_LOCATION_ID = anomaly_location[0]['location_id']
else:
    ANOMALY_LOCATION_ID = 1

anomaly_brand = spark.sql(f"""
  SELECT brand_id, SUM(brand_revenue) as rev
  FROM {CATALOG}.lakeflow.gold_brand_sales_day
  GROUP BY brand_id
  ORDER BY rev DESC
  LIMIT 1
""").collect()

if anomaly_brand:
    ANOMALY_BRAND_ID = anomaly_brand[0]['brand_id']
else:
    ANOMALY_BRAND_ID = 1

# Get the location name for answer validation
location_name_row = spark.sql(f"""
  SELECT name FROM {CATALOG}.simulator.locations WHERE location_id = {ANOMALY_LOCATION_ID}
""").collect()
ANOMALY_LOCATION_NAME = location_name_row[0]['name'] if location_name_row else f"Location {ANOMALY_LOCATION_ID}"

brand_name_row = spark.sql(f"""
  SELECT DISTINCT b.brand_id FROM {CATALOG}.lakeflow.gold_brand_sales_day b
  WHERE b.brand_id = {ANOMALY_BRAND_ID}
""").collect()

print(f"\U0001f50d Anomaly seeded at location_id={ANOMALY_LOCATION_ID} ({ANOMALY_LOCATION_NAME}), brand_id={ANOMALY_BRAND_ID}")

In [ ]:
# Level 1 view: revenue with an artificial drop at the anomaly location during evening hours
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.game.investigation_revenue_hourly AS
SELECT
  location_id,
  hour_ts,
  orders,
  CASE
    WHEN location_id = {ANOMALY_LOCATION_ID} AND HOUR(hour_ts) BETWEEN 18 AND 21
    THEN ROUND(revenue * 0.45, 2)
    ELSE revenue
  END AS revenue
FROM {CATALOG}.lakeflow.gold_location_sales_hourly
""")

# Level 1 view: brand sales with artificial drop for the target brand
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.game.investigation_brand_sales AS
SELECT
  brand_id,
  day,
  orders,
  items_sold,
  CASE
    WHEN brand_id = {ANOMALY_BRAND_ID}
    THEN ROUND(brand_revenue * 0.55, 2)
    ELSE brand_revenue
  END AS brand_revenue
FROM {CATALOG}.lakeflow.gold_brand_sales_day
""")

print("\u2705 Created Level 1 investigation views")

In [ ]:
# Level 2 view: delivery timing with artificial slowdown at the anomaly location
# Pick a specific date range where the "slowdown" occurred
slowdown_day_row = spark.sql(f"""
  SELECT DISTINCT order_day
  FROM {CATALOG}.lakeflow.gold_order_header
  ORDER BY order_day
  LIMIT 1 OFFSET 5
""").collect()

SLOWDOWN_START_DAY = str(slowdown_day_row[0]['order_day']) if slowdown_day_row else '2025-01-06'

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.game.investigation_delivery_times AS
WITH order_times AS (
  SELECT
    ae.order_id,
    ae.location_id,
    loc.name AS location_name,
    MIN(CASE WHEN ae.event_type = 'order_created' THEN try_to_timestamp(ae.ts) END) AS created_ts,
    MIN(CASE WHEN ae.event_type = 'gk_started' THEN try_to_timestamp(ae.ts) END) AS kitchen_start_ts,
    MIN(CASE WHEN ae.event_type = 'gk_finished' THEN try_to_timestamp(ae.ts) END) AS kitchen_end_ts,
    MIN(CASE WHEN ae.event_type = 'delivered' THEN try_to_timestamp(ae.ts) END) AS delivered_ts
  FROM {CATALOG}.lakeflow.all_events ae
  LEFT JOIN {CATALOG}.simulator.locations loc ON ae.location_id = loc.location_id
  GROUP BY ae.order_id, ae.location_id, loc.name
)
SELECT
  order_id,
  location_id,
  location_name,
  created_ts,
  TO_DATE(created_ts) AS order_day,
  ROUND(
    (UNIX_TIMESTAMP(kitchen_end_ts) - UNIX_TIMESTAMP(kitchen_start_ts)) / 60.0
    * CASE
        WHEN location_id = {ANOMALY_LOCATION_ID} AND TO_DATE(created_ts) >= '{SLOWDOWN_START_DAY}'
        THEN 2.8
        ELSE 1.0
      END,
    1
  ) AS kitchen_prep_minutes,
  ROUND(
    (UNIX_TIMESTAMP(delivered_ts) - UNIX_TIMESTAMP(created_ts)) / 60.0
    * CASE
        WHEN location_id = {ANOMALY_LOCATION_ID} AND TO_DATE(created_ts) >= '{SLOWDOWN_START_DAY}'
        THEN 1.8
        ELSE 1.0
      END,
    1
  ) AS total_delivery_minutes
FROM order_times
WHERE created_ts IS NOT NULL AND delivered_ts IS NOT NULL
""")

print(f"\u2705 Created Level 2 investigation view (slowdown starts {SLOWDOWN_START_DAY})")

##### Seed answer key

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

answers_schema = StructType([
    StructField("level", IntegerType()),
    StructField("question_key", StringType()),
    StructField("acceptable_answers", ArrayType(StringType())),
    StructField("hint", StringType()),
    StructField("max_score", IntegerType())
])

answers_data = [
    # Level 1: The Missing Orders (Genie quest)
    Row(
        level=1,
        question_key="location",
        acceptable_answers=[ANOMALY_LOCATION_NAME, str(ANOMALY_LOCATION_ID), f"location_id={ANOMALY_LOCATION_ID}", f"Location {ANOMALY_LOCATION_ID}"],
        hint="Try asking Genie: 'Which location has the lowest evening revenue?'",
        max_score=40
    ),
    Row(
        level=1,
        question_key="time_window",
        acceptable_answers=["evening", "18-21", "6pm-9pm", "dinner", "18:00-21:00", "6 pm to 9 pm", "dinner hours"],
        hint="Compare revenue across different hours of the day for each location.",
        max_score=30
    ),
    Row(
        level=1,
        question_key="brand",
        acceptable_answers=[str(ANOMALY_BRAND_ID), f"brand_id={ANOMALY_BRAND_ID}", f"Brand {ANOMALY_BRAND_ID}"],
        hint="Look at brand-level revenue. Which brand has an unusual dip?",
        max_score=30
    ),
    # Level 2: The Slow Kitchen (Dashboard quest)
    Row(
        level=2,
        question_key="location",
        acceptable_answers=[ANOMALY_LOCATION_NAME, str(ANOMALY_LOCATION_ID), f"location_id={ANOMALY_LOCATION_ID}", f"Location {ANOMALY_LOCATION_ID}"],
        hint="Compare average kitchen prep time across all locations.",
        max_score=50
    ),
    Row(
        level=2,
        question_key="date",
        acceptable_answers=[SLOWDOWN_START_DAY, SLOWDOWN_START_DAY.replace("-", "/")],
        hint="Look at when kitchen prep times started diverging from the norm.",
        max_score=50
    )
]

answers_df = spark.createDataFrame(answers_data, schema=answers_schema)
answers_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.game.quest_answers")
print(f"\u2705 Seeded {len(answers_data)} answers across 2 levels")

##### Seed quest metadata for the controller app

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.quest_levels (
  level INT,
  title STRING,
  subtitle STRING,
  story STRING,
  feature STRING,
  instructions STRING,
  question_keys ARRAY<STRING>
)
COMMENT 'Quest level definitions and narrative'
""")

levels_data = [
    Row(
        level=1,
        title="The Missing Orders",
        subtitle="Investigate the revenue drop using Genie",
        story="Orders are vanishing. Management says revenue is down at one location but nobody knows which one or why. You have access to Casper's data through Genie, a natural language interface. Ask the right questions to find: which location is affected, what time the drop happens, and which brand is hit hardest.",
        feature="Genie (AI/BI)",
        instructions="Open the Genie room and use natural language to query the data. Try questions like 'Show me revenue by location' or 'What are the hourly revenue trends?'. Find the anomaly and report back.",
        question_keys=["location", "time_window", "brand"]
    ),
    Row(
        level=2,
        title="The Slow Kitchen",
        subtitle="Find the bottleneck using AI/BI Dashboards",
        story="Customers are complaining about cold food. You suspect one kitchen is significantly slower than others. A dashboard has been prepared with delivery timing data. Use the filters and charts to identify which location has the problem and when it started.",
        feature="AI/BI Dashboards",
        instructions="Open the dashboard and explore the delivery timing charts. Compare kitchen prep times across locations. Use the date filter to find when the slowdown began. Report the affected location and the start date.",
        question_keys=["location", "date"]
    )
]

levels_schema = StructType([
    StructField("level", IntegerType()),
    StructField("title", StringType()),
    StructField("subtitle", StringType()),
    StructField("story", StringType()),
    StructField("feature", StringType()),
    StructField("instructions", StringType()),
    StructField("question_keys", ArrayType(StringType()))
])

levels_df = spark.createDataFrame(levels_data, schema=levels_schema)
levels_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.game.quest_levels")
print(f"\u2705 Seeded {len(levels_data)} quest levels")

In [ ]:
print(f"\n\U0001f3ae Game setup complete for catalog {CATALOG}!")
print(f"   Anomaly location: {ANOMALY_LOCATION_NAME} (id={ANOMALY_LOCATION_ID})")
print(f"   Anomaly brand: brand_id={ANOMALY_BRAND_ID}")
print(f"   Slowdown start: {SLOWDOWN_START_DAY}")
print(f"   Tables created: game.quest_state, game.quest_answers, game.quest_levels, game.leaderboard")
print(f"   Views created: game.investigation_revenue_hourly, game.investigation_brand_sales, game.investigation_delivery_times")